# Process and reduce columns here

In [2]:
import polars as pl
import os
from pathlib import Path
import duckdb as db


In [18]:
test_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/formatted_raw_datasets/2026-03-10eval21CelPaint_ChromaNoTreatDex.tsv")
test_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/formatted_raw_datasets/2026-03-11eval21CelPaint_PhenovNoTreatDex.tsv")

# grab columns

header = pl.scan_csv(test_path, separator="\t", truncate_ragged_lines=True)

print(len(header.columns))
# print(*header.columns, sep="\n")

display(header.head(10).collect())

4772


/tmp/ipykernel_36197/1490640213.py:8: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(len(header.columns))


Row,Column,Timepoint,Field,Object No,X,Y,Bounding Box,Position X [um],Position Y [um],Compound,Concentration,Cell Type,Cell Count,NoBorderObj - Nucleus Area [um2],NoBorderObj - Nucleus Roundness,NoBorderObj - Nucleus Perimeter [um],NoBorderObj - Nucleus Width [um],NoBorderObj - Nucleus Length [um],NoBorderObj - Nucleus Ratio Width to Length,NoBorderObj - Object No in Nuclei,"NoBorderObj - Object No in Nuclei bigger, rounder",NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Mean,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow StdDev,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Median,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Maximum,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Minimum,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Sum,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow CV [%],NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Quantile 90%,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Contrast,NoBorderObj - Nucleus Region Area [um2],NoBorderObj - Nucleus Region Roundness,NoBorderObj - Nucleus Region Perimeter [um],NoBorderObj - Nucleus Region Width [um],NoBorderObj - Nucleus Region Length [um],NoBorderObj - Nucleus Region Ratio Width to Length,…,NoBorderObj - Membrane Region PhenoVue647 Profile 5/5 SER-Saddle,NoBorderObj - Membrane Region PhenoVue647 Symmetry 02 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 03 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 04 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 05 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 12 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 13 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 14 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 15 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Threshold Compactness 30% SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Threshold Compactness 40% SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Threshold Compactness 50% SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Threshold Compactness 60% SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Axial Small Length SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Axial Length Ratio SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Radial Mean SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Radial Relative Deviation SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Radial Mean Ratio SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 1/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 2/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 3/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 4/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 5/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 SER Spot 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Hole 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Edge 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Ridge 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Valley 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Saddle 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Bright 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Dark 2 px,NoBorderObj - Membrane Region PhenoVue647 Haralick Correlation 2 px,NoBorderObj - Membrane Region PhenoVue647 Haralick Contrast 2 px,NoBorderObj - Membrane Region PhenoVue647 Haralick Sum Variance 2 px,NoBorderObj - Membrane Region PhenoVue647 Haralick Homogeneity 2 px,NoBorderObj - Membrane Region PhenoVue647 Gabor Min 2 px w2,NoBorderObj - Membrane Region PhenoVue647 Gabor Max 2 px w2
i64,i64,i64,i64,i64,i64,i64,str,f64,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,i64,i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
6,7,1,1,1,783,63,"""[7

In [25]:
with open(test_path, "r") as f:
    header_line = f.readline()
    expected_cols = len(header_line.split("\t"))
    print(f"Header has {expected_cols} columns")
    
    for i, line in enumerate(f, start=1):  # start=1 since we already read header
        n_fields = len(line.split("\t"))
        if n_fields > expected_cols:
            print(f"Row {i}: has {n_fields} fields (extra {n_fields - expected_cols})")
            print(f"Content: {line[:300]}")

Header has 4772 columns
Row 1: has 4773 fields (extra 1)
Content: 6	7	1	1	1	783	63	[755,48,801,77]	142.2	922.9					105.621	0.610423	59.6827	7.32467	18.8674	0.388218	6	2	494.213	53.9215	485	672	367	148758	10.9106	571	0.0490952	105.621	0.610423	59.6827	7.32467	18.8674	0.388218	0.203891	0.152904	0.0458963	0.079945	0.284943	0.202436	0.0417872	0.107347	0.715138	0.71513
Row 2: has 4773 fields (extra 1)
Content: 6	7	1	1	2	813	98	[801,86,826,110]	162.11	902.3					124.569	0.874059	45.2658	10.0372	16.0897	0.623831	8	3	959.335	162.426	939	1488	690	340564	16.9311	1198	0.337638	124.569	0.874059	45.2658	10.0372	16.0897	0.623831	0.177939	0.0386365	0.050944	0.0211403	0.263394	0.0658324	0.0723507	0.0360837	1.02756	1.
Row 3: has 4773 fields (extra 1)
Content: 6	7	1	1	3	301	102	[285,85,324,124]	-139.38	898.41					129.131	0.572338	70.383	5.9966	20.7017	0.289667	9	4	400.712	46.8386	392	565	303	147462	11.6889	460	0.237565	129.131	0.572338	70.383	5.9966	20.7017	0.289667	0.479265	0.142541	0.041223	0.207397	0.

KeyboardInterrupt: 

In [26]:
header = pl.read_csv(test_path, separator="\t", truncate_ragged_lines=False)

print(header.shape)
display(header)

(402037, 4772)


Row,Column,Timepoint,Field,Object No,X,Y,Bounding Box,Position X [um],Position Y [um],Compound,Concentration,Cell Type,Cell Count,NoBorderObj - Nucleus Area [um2],NoBorderObj - Nucleus Roundness,NoBorderObj - Nucleus Perimeter [um],NoBorderObj - Nucleus Width [um],NoBorderObj - Nucleus Length [um],NoBorderObj - Nucleus Ratio Width to Length,NoBorderObj - Object No in Nuclei,"NoBorderObj - Object No in Nuclei bigger, rounder",NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Mean,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow StdDev,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Median,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Maximum,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Minimum,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Sum,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow CV [%],NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Quantile 90%,NoBorderObj - Intensity Nucleus Region HOECHST 33342 narrow Contrast,NoBorderObj - Nucleus Region Area [um2],NoBorderObj - Nucleus Region Roundness,NoBorderObj - Nucleus Region Perimeter [um],NoBorderObj - Nucleus Region Width [um],NoBorderObj - Nucleus Region Length [um],NoBorderObj - Nucleus Region Ratio Width to Length,…,NoBorderObj - Membrane Region PhenoVue647 Profile 5/5 SER-Saddle,NoBorderObj - Membrane Region PhenoVue647 Symmetry 02 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 03 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 04 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 05 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 12 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 13 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 14 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Symmetry 15 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Threshold Compactness 30% SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Threshold Compactness 40% SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Threshold Compactness 50% SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Threshold Compactness 60% SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Axial Small Length SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Axial Length Ratio SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Radial Mean SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Radial Relative Deviation SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Radial Mean Ratio SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 1/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 2/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 3/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 4/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 Profile 5/5 SER-Edge,NoBorderObj - Membrane Region PhenoVue647 SER Spot 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Hole 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Edge 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Ridge 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Valley 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Saddle 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Bright 2 px,NoBorderObj - Membrane Region PhenoVue647 SER Dark 2 px,NoBorderObj - Membrane Region PhenoVue647 Haralick Correlation 2 px,NoBorderObj - Membrane Region PhenoVue647 Haralick Contrast 2 px,NoBorderObj - Membrane Region PhenoVue647 Haralick Sum Variance 2 px,NoBorderObj - Membrane Region PhenoVue647 Haralick Homogeneity 2 px,NoBorderObj - Membrane Region PhenoVue647 Gabor Min 2 px w2,NoBorderObj - Membrane Region PhenoVue647 Gabor Max 2 px w2
i64,i64,i64,i64,i64,i64,i64,str,f64,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,i64,i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
6,7,1,1,1,783,63,"""[7